# 01 — Preprocessing: From Waveform to Spectrogram

This notebook turns raw audio recordings into the magnitude spectrograms that feed the U-Net denoising autoencoder used throughout this project.

Before running the preprocessing scripts, it is worth explaining *why* the signal is represented the way it is. The choices made here — sampling rate, FFT size, window length, hop length, and which dataset to train on — all have direct consequences for what the model can learn and what hardware it can be trained on. The sections below walk through the signal-processing theory (Fourier Transform → DFT → FFT → STFT) and then justify every parameter used in `src/dataprocess/preprocess_SNSD.py` and `src/dataprocess/preprocess_VOiCES.py`.


## Theoretical Background: From Fourier Transform to Spectrogram

### Fourier Transform (FT)

The Fourier Transform decomposes a continuous-time signal $x(t)$ into the continuous frequencies that compose it:

$$X(f) = \int_{-\infty}^{\infty} x(t)\, e^{-j 2\pi f t}\, dt$$

This is the theoretical foundation, but it assumes an infinite, continuous signal — not something we can compute on a machine.

### Discrete Fourier Transform (DFT)

Audio is recorded as a finite sequence of discrete samples $x[n]$, $n = 0, \dots, N-1$ (16000 samples per second, in our case). The DFT is the discrete counterpart of the FT, decomposing a finite sequence into $N$ discrete frequency components:

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j 2\pi kn/N}, \qquad k = 0, \dots, N-1$$

Computed directly, this costs $O(N^2)$ operations — for a several-second audio clip at 16 kHz, that's tens of millions of samples, so a direct DFT is not practical.

### Fast Fourier Transform (FFT)

The FFT is not a different transform, it is an efficient *algorithm* (Cooley–Tukey being the classic example) that computes the exact same DFT result in $O(N \log N)$ operations, by recursively splitting the sum into even- and odd-indexed samples. The recursive splitting is cleanest when $N$ is a power of 2, which is why FFT-friendly implementations (`numpy`, `librosa`, `torch`) are fastest — and sometimes only defined — for power-of-2 window sizes. This is the reason `N_FFT` is chosen as a power of 2 later in this notebook.

### Short-Time Fourier Transform (STFT)

A single DFT over an entire audio clip would tell us *which* frequencies are present, but not *when* they occur — and speech is highly non-stationary (phonemes change constantly). The STFT solves this by sliding a short analysis window over the signal and computing a DFT/FFT on each windowed segment:

$$X[k, m] = \sum_{n=0}^{N-1} x[n + mH]\, w[n]\, e^{-j 2\pi kn/N}$$

where $w[n]$ is a window function, $N$ is the window/FFT size, $H$ is the hop length (samples between consecutive frames), and $m$ indexes the resulting frame. The output $X[k, m]$ is a complex-valued matrix — a **spectrogram** — with one axis for frequency bin $k$ and one axis for time frame $m$. This is exactly what `librosa.stft(...)` computes in `compute_mag_spectrogram` (see [utils.py](../src/utils.py)).


### Why Windowing Matters

Splitting the signal into frames is not as simple as slicing it into chunks. Cutting a waveform abruptly (a rectangular window) introduces sharp discontinuities at the edges of every frame. In the frequency domain, multiplying by a rectangular window corresponds to convolving the true spectrum with a sinc function, which "leaks" energy into frequencies that were never present in the original signal (spectral leakage).

To avoid this, we use a **Hann window**, which tapers smoothly to zero at both edges of the frame instead of cutting sharply. This removes the boundary artifacts, at the cost of down-weighting the samples near the edges of each frame — they contribute less to the analysis than the samples near the center.

That down-weighting is exactly why frames must **overlap**: if frames did not overlap, the down-weighted edges of the window would mean some parts of the signal are effectively under-represented. By hopping forward less than a full window length, every audio sample gets seen multiple times, near the center of some frame, so the loss of information at any single frame's edges is compensated by neighboring frames.


### Choosing the STFT Parameters

The same four parameters are used for both datasets processed in this notebook (`SAMPLE_RATE`, `N_FFT`, `WIN_LENGTH`, `HOP_LENGTH`). Each one was chosen deliberately:

| Parameter | Value | Rationale |
|---|---|---|
| `SAMPLE_RATE` | 16000 Hz | By the Nyquist theorem, this represents frequencies up to 8 kHz. Human speech intelligibility (fundamental + formants) lives almost entirely below that limit — 16 kHz is the standard "wideband" rate for speech tasks. Using it instead of music-grade rates (44.1/48 kHz) roughly triples the samples we'd need to process and store for no benefit on this task, keeping compute and storage requirements reasonable. |
| `N_FFT` | 512 | Number of samples per FFT, giving $N_{FFT}/2 + 1 = 257$ frequency bins (only non-negative frequencies are kept, since the input audio is real-valued) and a frequency resolution of $16000 / 512 \approx 31.25$ Hz per bin. Chosen as a power of 2 so the FFT algorithm can be applied at full efficiency (see above). |
| `WIN_LENGTH` | 512 | Analysis window duration of $512 / 16000 = 32$ ms. Speech-processing literature generally recommends 20–40 ms windows, matching the typical duration of a phoneme — long enough to capture a meaningful chunk of speech, short enough that the signal inside the window is approximately stationary. |
| `HOP_LENGTH` | 128 | $128 = 512 / 4$, i.e. 75% overlap between consecutive frames. As explained above, this compensates for the Hann window tapering the edges of each frame to near-zero, ensuring no part of the signal is under-represented. |

These parameters are passed as command-line arguments to the preprocessing scripts so they can be experimented with without touching the source code.


### Magnitude-Only Spectrograms: A Deliberate Trade-off

`compute_mag_spectrogram` (in [utils.py](../src/utils.py)) computes the complex STFT and keeps only its magnitude, `np.abs(D)`, discarding the phase entirely. This is a conscious simplification with a real cost: a complex spectrogram $D = |D|\, e^{j\phi}$ needs *both* magnitude and phase to be inverted back into a waveform exactly. By throwing away the phase, we introduce a reconstruction problem — turning a denoised magnitude spectrogram back into audio requires *estimating* a phase, which is what the Griffin-Lim algorithm (used in `play_denoised_sample`, [utils.py](../src/utils.py)) does iteratively, at the cost of some reconstruction quality.

The alternative — feeding the model both magnitude and phase — would double the number of channels in every spectrogram tensor. For a convolutional U-Net, that roughly doubles the activation memory and parameter count of every layer that touches the input. On an 8 GB VRAM GPU, that trade-off was not affordable alongside a batch size large enough for stable training, so magnitude-only was chosen as the practical option.


## Dataset Choice: VOiCES → SNSD

This project initially targeted the **VOiCES** dataset. After working with it, it became clear that VOiCES is built around far-field, reverberant recordings captured with varying microphone/room configurations — it is a better fit for **dereverberation** or robust ASR research than for the additive-noise denoising problem this project focuses on.

**SNSD** (Scalable Noisy Speech Dataset) was chosen instead: it provides clean/noisy speech pairs with controllable noise conditions and, importantly, lets us fix the clip duration. We standardized on **8-second** clips (`TARGET_FRAMES = 1024` frames in [preprocess_SNSD.py](../src/dataprocess/preprocess_SNSD.py), i.e. $1024 \times 128 / 16000 \approx 8s$) as a balance between giving the model enough temporal context and keeping spectrogram tensors small enough to train on consumer GPU hardware.

The VOiCES preprocessing pipeline below is kept for reference and was explored first, but **SNSD is the dataset used for all subsequent training, tuning, and evaluation notebooks.**


## Computing and Saving Spectrograms - VOiCES Dataset (exploratory, kept for reference)


In [2]:
# === CONFIGURATION ===
DATASET_ROOT = "/media/gabriel/'500 BKP'/datasets/VOICES-DataSet/VOiCES_devkit"
OUT_DIR = "/media/gabriel/'500 BKP'/datasets/VOICES-DataSet/preprocessed_complex"

N_FFT = 512
HOP_LENGTH = 128
WIN_LENGTH = 512
SAMPLE_RATE = 16000

In [3]:
!python ../src/dataprocess/preprocess_VOiCES.py {DATASET_ROOT} {OUT_DIR} {N_FFT} {HOP_LENGTH} {WIN_LENGTH}

[test] Processing speakers: 100%|█████████████| 100/100 [00:21<00:00,  4.62it/s]


## Computing and Saving Spectrograms - SNSD Dataset (primary dataset)


In [5]:
# === CONFIGURATION ===
DATASET_ROOT = "/media/gabriel/'500 BKP'/datasets/SNSD-DataSet/raw"
OUT_DIR = "/media/gabriel/'500 BKP'/datasets/SNSD-DataSet/preprocessed"

N_FFT = 512
HOP_LENGTH = 128
WIN_LENGTH = 512
SAMPLE_RATE = 16000

In [6]:
!python ../src/dataprocess/preprocess_SNSD.py {DATASET_ROOT} {OUT_DIR} {N_FFT} {HOP_LENGTH} {WIN_LENGTH}

[test] Processing noisy files: 100%|████████| 1205/1205 [00:21<00:00, 55.88it/s]


## From Saved Spectrograms to a Torch Dataset

The preprocessing scripts above save each clip's magnitude spectrogram to disk as a `.npy` file (one per clean/noisy recording). Loading an entire dataset of spectrograms into memory at once is not practical, so we wrap the saved files in a PyTorch `Dataset` — `DenoisingSpectrogramDataset` in [torch_dataset_SNSD.py](../src/dataprocess/torch_dataset_SNSD.py).

Implementing PyTorch's `Dataset` interface (`__len__` and `__getitem__`) lets a `DataLoader` fetch and load only one batch at a time from disk during training, rather than holding the full dataset in memory. Each pair of clean/noisy spectrograms is also padded or randomly cropped to a fixed shape (`target_bins × target_frames`, 257×1024 for our 8-second SNSD clips) so that samples can be batched into a single tensor. Batch size and the rest of the `DataLoader` configuration are covered in the training and hyperparameter-tuning notebooks — here we only care that the data is in a shape the model can consume.


In [ ]:
import sys
sys.path.append('..')

from src.dataprocess.torch_dataset_SNSD import DenoisingSpectrogramDataset

train_set = DenoisingSpectrogramDataset(OUT_DIR, split='train')
noisy, clean = train_set[0]

print(f"Samples in training set: {len(train_set)}")
print(f"Noisy spectrogram shape: {tuple(noisy.shape)}")
print(f"Clean spectrogram shape: {tuple(clean.shape)}")